# 🔮 Predictur — Pipeline Walkthrough

Notebook para ejecutar el pipeline de pronóstico **paso a paso**, inspeccionando los resultados intermedios en cada etapa.

**Flujo completo:**
```
1. Carga de datos        →  load_series() + load_full_frame()
2. Regresores exógenos   →  build_exog()
3. CV walk-forward       →  cross_validate() × 3 modelos
4. Ensemble ponderado    →  inverse_mape_weights() + build_ensemble_predictions()
5. Métricas              →  summarize_cv()
6. Pronóstico futuro     →  run_forecast()
7. Guardar artefactos    →  reports/
```

> **Nota:** Ejecutar desde la raíz del proyecto o desde `notebooks/`. El paquete resuelve rutas absolutas automáticamente.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import warnings
warnings.filterwarnings("ignore")

# Asegurar que src/ esté en el path (necesario si se ejecuta desde notebooks/)
PROJECT_ROOT = Path("__file__").resolve().parent.parent
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

# Importar módulos del paquete
from predictur.data import load_series, load_full_frame
from predictur.features import build_exog
from predictur.evaluation import cross_validate, summarize_cv, walk_forward_splits
from predictur.models import SarimaForecaster, ETSForecaster, ProphetForecaster
from predictur.models.ensemble import inverse_mape_weights, build_ensemble_predictions
from predictur.pipeline import run_pipeline
from predictur.forecast import run_forecast

print("✅ Imports OK")

---
## 1. Carga de datos

Cargamos `data/processed/master_tourism_series.csv` con `load_series()` (extrae solo `Ocupacion_Caribe`) y con `load_full_frame()` (todas las columnas).

In [ ]:
# Serie objetivo: tasa de ocupación hotelera Caribe
y = load_series()
print(f"Serie: {y.name}")
print(f"Período: {y.index[0].date()} → {y.index[-1].date()}")
print(f"Observaciones: {len(y)}")
print(f"Min: {y.min():.2f}%   Max: {y.max():.2f}%   Media: {y.mean():.2f}%")
y.tail(6)

In [ ]:
# Marco completo (todas las columnas disponibles)
df_full = load_full_frame()
print(f"Columnas disponibles: {list(df_full.columns)}")
df_full.head(3)

In [ ]:
fig, ax = plt.subplots()
ax.plot(y.index, y.values, color="#2563EB", linewidth=2, label="Ocupación Caribe")
ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2021-06-01"),
           alpha=0.15, color="red", label="COVID shock")
ax.axvspan(pd.Timestamp("2021-07-01"), pd.Timestamp("2021-12-01"),
           alpha=0.15, color="orange", label="Recuperación")
ax.set_title("Tasa de Ocupación Hotelera — Región Caribe", fontweight="bold")
ax.set_ylabel("Ocupación (%)")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

---
## 2. Regresores exógenos

`build_exog()` construye la matriz de regresores a partir del índice temporal: flags COVID, Semana Santa y Carnaval de Barranquilla.

In [ ]:
exog = build_exog(y.index)
print(f"Regresores: {list(exog.columns)}")
print(f"Shape: {exog.shape}")
exog.head(8)

In [ ]:
fig, axes = plt.subplots(len(exog.columns), 1,
                         figsize=(12, 2 * len(exog.columns)), sharex=True)
colors = ["#DC2626", "#F97316", "#7C3AED", "#059669"]
for ax, col, color in zip(axes, exog.columns, colors):
    ax.fill_between(exog.index, exog[col], alpha=0.7, color=color, step="pre")
    ax.set_ylabel(col, fontsize=9)
    ax.set_ylim(-0.1, 1.3)
    ax.set_yticks([0, 1])
axes[0].set_title("Regresores exógenos", fontweight="bold")
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.tight_layout()
plt.show()

---
## 3. Walk-forward CV — estructura de folds

Antes de correr los modelos, visualizamos cómo quedan los folds de la validación cruzada.

Parámetros por defecto:
- `initial_train = 48` meses de entrenamiento inicial
- `horizon = 6` meses de predicción por fold
- `step = 1` mes de avance entre folds

In [ ]:
INITIAL_TRAIN = 48
HORIZON = 6
STEP = 1

folds = list(walk_forward_splits(y.index, initial_train=INITIAL_TRAIN,
                                  horizon=HORIZON, step=STEP))
print(f"Total folds: {len(folds)}")
print(f"Observaciones evaluadas por fold: {HORIZON}")
print(f"Total observaciones evaluadas: {len(folds) * HORIZON}")
print()
print(f"Primer fold → train hasta: {folds[0].train_end.date()} | test: {folds[0].test_start.date()} → {folds[0].test_end.date()}")
print(f"Último fold  → train hasta: {folds[-1].train_end.date()} | test: {folds[-1].test_start.date()} → {folds[-1].test_end.date()}")

In [ ]:
# Visualizar los primeros 10 folds
fig, ax = plt.subplots(figsize=(12, 3))
for i, fold in enumerate(folds[:10]):
    ax.barh(i, (fold.train_end - y.index[0]).days, left=mdates.date2num(y.index[0]),
            height=0.6, color="#3B82F6", alpha=0.7)
    ax.barh(i, (fold.test_end - fold.test_start).days, left=mdates.date2num(fold.test_start),
            height=0.6, color="#F59E0B", alpha=0.9)
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.set_xlabel("Fecha")
ax.set_ylabel("Fold")
ax.set_title("Walk-forward CV — primeros 10 folds (azul=train, naranja=test)", fontweight="bold")
plt.tight_layout()
plt.show()

---
## 4. CV por modelo

Ejecutamos `cross_validate()` para cada modelo por separado para inspeccionar los resultados individualmente.

In [ ]:
print("=" * 50)
print("SARIMAX — Walk-forward CV")
print("=" * 50)
cv_sarima = cross_validate(
    lambda: SarimaForecaster(), y, exog=exog,
    initial_train=INITIAL_TRAIN, horizon=HORIZON, step=STEP,
    verbose=True,
)
print(f"\nResultado: {cv_sarima.shape[0]} filas")
cv_sarima.head()

In [ ]:
print("=" * 50)
print("ETS (Holt-Winters) — Walk-forward CV")
print("=" * 50)
cv_ets = cross_validate(
    lambda: ETSForecaster(), y, exog=exog,
    initial_train=INITIAL_TRAIN, horizon=HORIZON, step=STEP,
    verbose=True,
)
print(f"\nResultado: {cv_ets.shape[0]} filas")

In [ ]:
print("=" * 50)
print("Prophet — Walk-forward CV")
print("=" * 50)
cv_prophet = cross_validate(
    lambda: ProphetForecaster(), y, exog=exog,
    initial_train=INITIAL_TRAIN, horizon=HORIZON, step=STEP,
    verbose=True,
)
print(f"\nResultado: {cv_prophet.shape[0]} filas")

---
## 5. Ensemble ponderado por inverso del MAPE

Combinamos los tres modelos con pesos proporcionales al inverso de su MAPE en CV. Mejor modelo → mayor peso.

In [ ]:
# Concatenar predicciones base
base = pd.concat([cv_sarima, cv_ets, cv_prophet], ignore_index=True)
print(f"Predicciones base totales: {len(base)} filas")

# Calcular pesos
weights = inverse_mape_weights(base)
print("\nPesos del ensemble (inverso-MAPE):")
for model, w in sorted(weights.items(), key=lambda x: -x[1]):
    print(f"  {model:<10} {w:.4f}  ({w*100:.1f}%)")

In [ ]:
# Construir predicciones ensemble
ensemble = build_ensemble_predictions(base, weights=weights)
full = pd.concat([base, ensemble], ignore_index=True)
print(f"DataFrame completo (base + ensemble): {len(full)} filas")
print(f"Modelos: {full['model'].unique().tolist()}")
full[full.model == "Ensemble"].head()

---
## 6. Métricas de evaluación

In [ ]:
metrics = summarize_cv(full)
print("=" * 55)
print("MÉTRICAS — Walk-forward CV (ordenadas por MAPE)")
print("=" * 55)
print(metrics.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
bar_colors = ["#F59E0B" if m == "Ensemble" else "#6B7280" for m in metrics["model"]]

for ax, metric in zip(axes, ["MAE", "RMSE", "MAPE"]):
    bars = ax.bar(metrics["model"], metrics[metric], color=bar_colors, edgecolor="white", linewidth=0.5)
    ax.set_title(metric, fontweight="bold")
    ax.set_ylabel("Puntos porcentuales" if metric != "MAPE" else "%")
    for bar, val in zip(bars, metrics[metric]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                f"{val:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax.set_ylim(0, metrics[metric].max() * 1.25)

plt.suptitle("Comparación de modelos — Walk-forward CV", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Predicciones vs real por modelo
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
model_colors = {"SARIMAX": "#3B82F6", "ETS": "#10B981", "Prophet": "#8B5CF6", "Ensemble": "#F59E0B"}

for ax, model_name in zip(axes.flat, ["SARIMAX", "ETS", "Prophet", "Ensemble"]):
    subset = full[full.model == model_name].copy()
    subset = subset.sort_values("Date")
    ax.plot(subset["Date"], subset["y_true"], color="black", linewidth=1.2,
            label="Real", alpha=0.7)
    ax.plot(subset["Date"], subset["y_pred"], color=model_colors[model_name],
            linewidth=1.5, label=model_name, linestyle="--")
    mape_val = metrics.loc[metrics.model == model_name, "MAPE"].values[0]
    ax.set_title(f"{model_name} (MAPE={mape_val:.2f}%)", fontweight="bold")
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.suptitle("Predicciones CV vs Real", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()

---
## 7. Guardar artefactos de CV en `reports/`

In [ ]:
from predictur.data import PROJECT_ROOT

REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

full.to_csv(REPORTS_DIR / "cv_predictions.csv", index=False)
metrics.to_csv(REPORTS_DIR / "metrics.csv", index=False)
pd.Series(weights, name="weight").to_csv(REPORTS_DIR / "ensemble_weights.csv")

print(f"✅ Guardado en {REPORTS_DIR}")
for f in sorted(REPORTS_DIR.iterdir()):
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")

---
## 8. Pronóstico futuro (12 meses)

`run_forecast()` entrena cada modelo sobre la **serie completa** y genera el pronóstico con intervalos de confianza al 80% y 95%.

In [ ]:
FORECAST_HORIZON = 12  # meses

forecast_df = run_forecast(
    horizon=FORECAST_HORIZON,
    verbose=True,
)
print(f"\nShape: {forecast_df.shape}")
print(f"Modelos: {forecast_df['model'].unique().tolist()}")
forecast_df[forecast_df.model == "Ensemble"]

In [ ]:
# Visualización del pronóstico ensemble con intervalos de confianza
ens = forecast_df[forecast_df.model == "Ensemble"].copy()

fig, ax = plt.subplots(figsize=(13, 5))

# Historia reciente (últimos 24 meses)
history = y[-24:]
ax.plot(history.index, history.values, color="#1E293B", linewidth=2.5,
        label="Histórico", zorder=3)

# Pronósticos base
base_colors = {"SARIMAX": "#3B82F6", "ETS": "#10B981", "Prophet": "#8B5CF6"}
for model_name, color in base_colors.items():
    sub = forecast_df[forecast_df.model == model_name]
    ax.plot(sub["Date"], sub["yhat"], color=color, linewidth=1.2,
            linestyle="--", alpha=0.6, label=model_name)

# Ensemble: punto + bandas
ax.plot(ens["Date"], ens["yhat"], color="#F59E0B", linewidth=2.5,
        label="Ensemble", zorder=4)
ax.fill_between(ens["Date"], ens["yhat_lower_80"], ens["yhat_upper_80"],
                color="#F59E0B", alpha=0.25, label="IC 80%")
ax.fill_between(ens["Date"], ens["yhat_lower_95"], ens["yhat_upper_95"],
                color="#F59E0B", alpha=0.10, label="IC 95%")

# Línea divisoria historia/pronóstico
ax.axvline(y.index[-1], color="gray", linestyle=":", linewidth=1.5, label="Hoy")

ax.set_title(f"Pronóstico de ocupación hotelera Caribe — {FORECAST_HORIZON} meses",
             fontweight="bold")
ax.set_ylabel("Ocupación (%)")
ax.legend(loc="lower left", fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

---
## 9. Pipeline completo en un solo comando

Las celdas anteriores desglosan cada paso. Si solo quieres ejecutar todo de una vez, usa `run_pipeline()` seguido de `run_forecast()`.

In [ ]:
# ⚠️  Esto re-ejecuta el CV completo (~33 folds × 3 modelos). Puede tardar varios minutos.
# Descomenta para correr:

# results = run_pipeline(verbose=True)
# forecast_df = run_forecast(horizon=12, verbose=True)
#
# print("\n✅ Pipeline completado")
# print(results["metrics"].to_string(index=False))

---
## 10. Leer artefactos generados

Inspecciona los CSVs guardados en `reports/` sin re-ejecutar el pipeline.

In [ ]:
# Métricas guardadas
pd.read_csv(REPORTS_DIR / "metrics.csv")

In [ ]:
# Pesos del ensemble
pd.read_csv(REPORTS_DIR / "ensemble_weights.csv", index_col=0)

In [ ]:
# Pronóstico futuro
fc = pd.read_csv(REPORTS_DIR / "forecast.csv", parse_dates=["Date"])
fc[fc.model == "Ensemble"][["Date", "yhat", "yhat_lower_95", "yhat_upper_95", "horizon"]]